# CropCop Track B — EAAI External Validation

Paper-first cross-source evaluation on GVLiD v5 and Irish Potato using the frozen R07 S1/S2/S3 checkpoints. This notebook intentionally excludes the retired DINO/ORB qualification machinery.

**Attach exactly two existing Notebook-00 datasets:** the Track-B infrastructure bundle and the Track-B external bundle. Enable one Kaggle GPU and Internet. No Kaggle secret is required.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys

EXPECTED_SOURCE_COMMIT = 'd9e9a323834ae6d2d6b745e18e4023f8789a6c7e'
REPO = Path('/kaggle/working/ResearchWork-CropCop-v7')
OUTPUT = Path('/kaggle/working/trackb_eaai_results')

if OUTPUT.exists() and any(OUTPUT.iterdir()):
    raise RuntimeError(f'Refusing to overwrite non-empty output: {OUTPUT}')

manifests = sorted(Path('/kaggle/input').glob('**/TRACKB_INPUT_MANIFEST.json'))
roles = []
for path in manifests:
    obj = json.loads(path.read_text(encoding='utf-8'))
    roles.append(str(obj.get('role', '')))
expected_roles = {'core','historical_compare','gvlid_v5','irish_potato'}
if set(roles) != expected_roles or len(roles) != 4:
    raise RuntimeError(f'Attach exactly one of each Track-B role; observed={roles}')
print({'status':'PASS_ATTACHED_ROLES','roles':sorted(roles)})

if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git','init',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'remote','add','origin','https://github.com/rana-m-ahmed/ResearchWork-CropCop.git'],check=True)
subprocess.run(['git','-C',str(REPO),'fetch','--depth','1','origin',EXPECTED_SOURCE_COMMIT],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach','FETCH_HEAD'],check=True)
observed = subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
if observed != EXPECTED_SOURCE_COMMIT:
    raise RuntimeError(f'Source commit mismatch: {observed}')
print({'status':'PASS_PINNED_SOURCE','commit':observed})


In [ ]:
import importlib.metadata as md

required = {
    'torch': '2.12.1',
    'torchvision': '0.27.1',
    'numpy': '2.5.2',
    'Pillow': '12.3.0',
}
observed = {}
missing_or_drift = []
for package, version in required.items():
    try:
        current = md.version(package)
    except md.PackageNotFoundError:
        current = None
    observed[package] = current
    if current != version:
        missing_or_drift.append(f'{package}=={version}')
print({'runtime_before': observed})
if missing_or_drift:
    print('Repairing only the model-critical frozen packages:', missing_or_drift)
    subprocess.run([
        sys.executable,'-m','pip','install','--disable-pip-version-check','--no-cache-dir','--upgrade',
        *missing_or_drift,
    ],check=True)
else:
    print('Model-critical runtime already matches the frozen stack.')


In [ ]:
subprocess.run(['nvidia-smi'],check=True)

runner = REPO / 'journal_extension/scripts/run_trackb_eaai_external_validation.py'
protocol = REPO / 'journal_extension/track_b_r07/TRACKB_EAAI_PROTOCOL_v1.json'
cmd = [
    sys.executable, '-B', str(runner),
    '--input-root', '/kaggle/input',
    '--output-root', str(OUTPUT),
    '--protocol', str(protocol),
    '--mode', 'all',
    '--device', 'cuda:0',
    '--batch-size', '64',
    '--audit-workers', '8',
    '--loader-workers', '4',
]
print('Launching:', ' '.join(cmd), flush=True)
subprocess.run(cmd,cwd=REPO,check=True,env={**os.environ,'PYTHONDONTWRITEBYTECODE':'1'})


In [ ]:
final_path = OUTPUT / 'TRACKB_FINAL_MANIFEST.json'
if not final_path.is_file():
    raise RuntimeError('Final Track-B manifest missing')
final = json.loads(final_path.read_text(encoding='utf-8'))
if final.get('status') != 'PASS_TRACKB_EAAI_EXTERNAL_VALIDATION':
    raise RuntimeError(f'Unexpected terminal status: {final.get("status")}')
if final.get('v1_test_accessed') is not False:
    raise RuntimeError('V1 test closure was not preserved')

archive_base = Path('/kaggle/working/CropCop_TrackB_EAAI_External_Validation')
archive = Path(shutil.make_archive(str(archive_base),'zip',root_dir=OUTPUT))
print(json.dumps({
    'status': final['status'],
    'external_prediction_count': final['external_prediction_count'],
    'manifest_sha256': final['manifest_sha256'],
    'results_dir': str(OUTPUT),
    'evidence_zip': str(archive),
},indent=2,sort_keys=True))
